# 1. Synthetic Data Generation & Exploration

Generate synthetic animals using the BE model, visualise behaviour across
learning phases, and build intuition for how parameters shape behaviour.

**Sections:**
1. Setup & imports
2. Single session: model mechanics
3. Parameter sweeps: how each param affects behaviour
4. Multi-session trajectories: naive → expert → shift
5. Cycling design: A→B→A→B with meta-learning
6. Distribution effects: Uniform vs Hard-A vs Hard-B

## 1. Setup

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# --- Project imports ---
from Models.BE_core import BEParams, BEState, BEModel, ModelTrace
from Models.BE_model import BoundaryEstimationModel
from Helpers.psychometry import fit_psychometric
from Helpers.utils import cumulative_gaussian
from Analysis.summary_stats import compute_summary_stats, list_available_stats, get_stat_names_expanded
from Analysis.update_matrix import compute_update_matrix, compute_update_matrix_from_model_trace
from Data.structures import (
    generate_synthetic_animal, generate_synthetic_session,
    sample_stimuli, stimulus_to_category,
    DistributionEpoch, make_distribution_schedule,
    param_trajectory_naive_to_expert, param_trajectory_full, param_trajectory_cycling,
)

print("Available summary stats:", list_available_stats())

## 2. Single Session: Model Mechanics

Simulate one session with fixed parameters and visualise:
- Trial-by-trial stimuli and choices
- Belief distribution evolution
- Psychometric curve
- Update matrix

In [ ]:
# --- Configuration ---
PARAMS = dict(sigma_percep=0.2, A_repulsion=0.25, eta_learning=0.1, eta_relax=0.01)
N_TRIALS = 500
BURN_IN = 100
SEED = 42

## 3. Parameter Sweeps

Vary one parameter at a time while holding others constant.
See how each parameter shapes the psychometric curve and summary stats.

In [ ]:
def sweep_parameter(param_name, values, base_params, n_trials=400, burn_in=500, seed=42):
    """Sweep one parameter, return list of (params, choices, stimuli, categories, trace)."""
    results = []
    for val in values:
        p = dict(base_params)
        p[param_name] = val
        
        model = BoundaryEstimationModel(**p)
        model.reset_belief(burn_in=burn_in, burn_in_seed=seed)
        
        rng = np.random.default_rng(seed)
        stim = sample_stimuli(n_trials, 'Uniform', rng)
        cats = stimulus_to_category(stim)
        ch, pB = model.simulate_session(stim, cats, rng=rng, store_history=True)
        
        results.append({
            'value': val,
            'params': p,
            'choices': ch,
            'stimuli': stim,
            'categories': cats,
            'p_B': pB,
            'trace': model.get_model_trace(),
        })
    return results


BASE = dict(sigma_percep=0.15, A_repulsion=0.10, eta_learning=0.30, eta_relax=0.12)

sweeps = {
    'sigma_percep': np.array([0.05, 0.10, 0.15, 0.25, 0.40]),
    'A_repulsion':  np.array([0.0, 0.05, 0.10, 0.20, 0.40]),
    'eta_learning': np.array([0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 0.8]),
    'eta_relax':    np.array([0.00001, 0.0001, 0.001, 0.01, 0.01]),
}

sweep_results = {}
for pname, vals in sweeps.items():
    sweep_results[pname] = sweep_parameter(pname, vals, BASE)
    print(f"  {pname}: done")

In [ ]:
# --- Psychometric curves across sweeps ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
x_fine = np.linspace(-1, 1, 200)
cmap = plt.cm.viridis

for ax, pname in zip(axes.flat, sweeps.keys()):
    results = sweep_results[pname]
    vals = sweeps[pname]
    
    for i, res in enumerate(results):
        color = cmap(i / (len(results) - 1))
        valid = ~np.isnan(res['choices'])
        psych = fit_psychometric(res['stimuli'][valid], res['choices'][valid])
        
        if psych['success']:
            y_fit = cumulative_gaussian(x_fine, psych['mu'], psych['sigma'],
                                         psych['lapse_low'], psych['lapse_high'])
            ax.plot(x_fine, y_fit, color=color, linewidth=2, 
                    label=f"{pname}={res['value']:.2f}")
    
    ax.axhline(0.5, color='grey', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Stimulus')
    ax.set_ylabel('P(choose B)')
    ax.set_title(f'Sweep: {pname}')
    ax.legend(fontsize=7)
    ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# --- Summary stats across sweeps ---
stat_names_for_sweep = ['accuracy', 'recency', 'win_stay', 'stimulus_sensitivity', 'choice_entropy']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, pname in zip(axes.flat, sweeps.keys()):
    results = sweep_results[pname]
    vals = [r['value'] for r in results]
    
    for sname in stat_names_for_sweep:
        stat_vals = []
        for res in results:
            valid = ~np.isnan(res['choices'])
            s = compute_summary_stats(
                res['choices'][valid], res['stimuli'][valid], res['categories'][valid],
                stat_names=[sname], return_dict=True,
            )
            # Handle dict returns (psychometric)
            if isinstance(s[sname], dict):
                stat_vals.append(list(s[sname].values())[0])
            else:
                stat_vals.append(float(s[sname]))
        
        ax.plot(vals, stat_vals, 'o-', linewidth=1.5, markersize=5, label=sname)
    
    ax.set_xlabel(pname)
    ax.set_ylabel('Stat value')
    ax.set_title(f'Stats vs {pname}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 4. Multi-Session Trajectories

Generate synthetic animals with full learning trajectories and
visualise how behaviour evolves across sessions.

In [ ]:
# --- 4a. Naive -> Expert (simple) ---
N_SESSIONS = 25
params_simple = param_trajectory_naive_to_expert(N_SESSIONS)

animal_simple, gt_simple = generate_synthetic_animal(
    animal_id='SYN_SIMPLE',
    true_params=params_simple,
    trials_per_session=300,
    seed=42,
)

print(f"Generated {animal_simple.n_sessions} sessions")
print(f"eta trajectory: {[f'{e:.3f}' for e in gt_simple['params_per_session']['eta_learning'][:5]]} "
      f"... {[f'{e:.3f}' for e in gt_simple['params_per_session']['eta_learning'][-3:]]}")

In [ ]:
# --- Performance and key stats across sessions ---
from Analysis.summary_stats import compute_summary_stats

def compute_trajectory_stats(animal, gt, stat_names=None):
    """Compute stats per session for a synthetic animal."""
    if stat_names is None:
        stat_names = ['accuracy', 'psychometric', 'recency', 'win_stay',
                      'stimulus_sensitivity', 'choice_entropy']
    
    results = {'session_idx': [], 'eta_learning': [], 'distribution': []}
    # Initialise stat columns
    example = compute_summary_stats(
        np.array([0,1,0,1.0]), np.array([-0.5,0.5,-0.3,0.3]),
        np.array([0,1,0,1]), stat_names=stat_names, return_dict=True
    )
    for sname, val in example.items():
        if isinstance(val, dict):
            for k in val:
                results[k] = []
        else:
            results[sname] = []
    
    for s_idx, session in enumerate(animal.sessions):
        arrays = session.trials.get_model_arrays(exclude_abort=True, exclude_opto=True)
        valid = ~arrays['no_response']
        
        results['session_idx'].append(s_idx)
        results['eta_learning'].append(gt['params_per_session']['eta_learning'][s_idx])
        results['distribution'].append(session.distribution)
        
        stats = compute_summary_stats(
            arrays['choices'][valid], arrays['stimuli'][valid], arrays['categories'][valid],
            stat_names=stat_names, return_dict=True
        )
        for sname, val in stats.items():
            if isinstance(val, dict):
                for k, v in val.items():
                    results[k].append(float(v))
            else:
                results[sname].append(float(val))
    
    return {k: np.array(v) for k, v in results.items()}

traj_simple = compute_trajectory_stats(animal_simple, gt_simple)

In [ ]:
# --- Plot trajectory ---
def plot_trajectory(traj, title='', dist_changes=None):
    """Plot key stats across sessions with eta overlay."""
    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    s = traj['session_idx']
    
    plots = [
        ('accuracy', 'Accuracy', '#1976d2'),
        ('recency', 'Recency index', '#d32f2f'),
        ('pse', 'PSE (mu)', '#388e3c'),
        ('slope', 'Psychometric slope (sigma)', '#f57c00'),
        ('win_stay', 'Win-stay rate', '#7b1fa2'),
        ('stimulus_sensitivity', 'Stimulus sensitivity', '#00796b'),
    ]
    
    for ax, (key, label, color) in zip(axes.flat, plots):
        if key in traj:
            ax.plot(s, traj[key], 'o-', color=color, linewidth=1.5, markersize=4, label=label)
            ax.set_ylabel(label, fontsize=9)
        
        # Overlay eta on twin axis
        ax2 = ax.twinx()
        ax2.plot(s, traj['eta_learning'], '--', color='grey', alpha=0.5, linewidth=1)
        ax2.set_ylabel('eta', color='grey', fontsize=8)
        ax2.tick_params(axis='y', colors='grey', labelsize=7)
        
        # Mark distribution changes
        if dist_changes:
            for dc in dist_changes:
                ax.axvline(dc, color='red', linewidth=1, linestyle=':', alpha=0.7)
    
    axes[-1, 0].set_xlabel('Session')
    axes[-1, 1].set_xlabel('Session')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    return fig

fig = plot_trajectory(traj_simple, title='Naive → Expert (Uniform distribution)')
plt.show()

### 4b. Full trajectory: Naive → Expert → Distribution Shift

In [ ]:
# --- Full trajectory with distribution shift ---
N_NAIVE, N_EXPERT, N_POST = 15, 5, 10
params_full = param_trajectory_full(
    n_sessions_naive=N_NAIVE,
    n_sessions_expert=N_EXPERT,
    n_sessions_post_shift=N_POST,
)
schedule_full = make_distribution_schedule([
    DistributionEpoch('Uniform', N_NAIVE + N_EXPERT),
    DistributionEpoch('Hard-A', N_POST),
])

animal_full, gt_full = generate_synthetic_animal(
    animal_id='SYN_FULL',
    true_params=params_full,
    distribution_schedule=schedule_full,
    trials_per_session=300,
    seed=123,
)

traj_full = compute_trajectory_stats(animal_full, gt_full)
fig = plot_trajectory(
    traj_full,
    title='Full trajectory: Naive → Expert → Hard-A shift',
    dist_changes=[N_NAIVE + N_EXPERT],
)
plt.show()

## 5. Cycling Design: A→B→A→B with Meta-Learning

In [ ]:
# --- Cycling design ---
N_BASELINE = 15
N_PER_CYCLE = 5
N_CYCLES = 4

params_cyc = param_trajectory_cycling(
    n_sessions_baseline=N_BASELINE,
    n_sessions_per_cycle=N_PER_CYCLE,
    n_cycles=N_CYCLES,
)
schedule_cyc = make_distribution_schedule([
    DistributionEpoch('Uniform', N_BASELINE),
    DistributionEpoch('Hard-A', N_PER_CYCLE),
    DistributionEpoch('Hard-B', N_PER_CYCLE),
    DistributionEpoch('Hard-A', N_PER_CYCLE),
    DistributionEpoch('Hard-B', N_PER_CYCLE),
])

animal_cyc, gt_cyc = generate_synthetic_animal(
    animal_id='SYN_CYCLE',
    true_params=params_cyc,
    distribution_schedule=schedule_cyc,
    trials_per_session=250,
    seed=456,
)

traj_cyc = compute_trajectory_stats(animal_cyc, gt_cyc)

# Mark all distribution transitions
shift_points = [N_BASELINE + i * N_PER_CYCLE for i in range(N_CYCLES)]
fig = plot_trajectory(
    traj_cyc,
    title='Cycling design: Uniform → Hard-A → Hard-B → Hard-A → Hard-B',
    dist_changes=shift_points,
)
plt.show()

In [ ]:
# --- Distribution labels per session ---
fig, ax = plt.subplots(figsize=(14, 4))
dists = traj_cyc['distribution']
eta = traj_cyc['eta_learning']
s = traj_cyc['session_idx']

dist_colors = {'Uniform': '#1976d2', 'Hard-A': '#d32f2f', 'Hard-B': '#388e3c'}

for i in range(len(s)):
    ax.bar(s[i], eta[i], color=dist_colors.get(dists[i], 'grey'), alpha=0.7, width=0.8)

ax.set_xlabel('Session')
ax.set_ylabel('eta_learning')
ax.set_title('Learning rate trajectory with distribution labels')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.7, label=d) for d, c in dist_colors.items()]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()

## 6. Distribution Effects

Compare behaviour under Uniform, Hard-A, and Hard-B distributions
for the same model parameters. Shows how the stimulus distribution
shapes the psychometric curve and summary stats even without
any parameter changes.

In [ ]:
# --- Compare distributions with identical parameters ---
EXPERT_PARAMS = dict(sigma_percep=0.15, A_repulsion=0.30, eta_learning=0.10, eta_relax=0.001)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x_fine = np.linspace(-1, 1, 200)

for ax, dist_name in zip(axes, ['Uniform', 'Hard-A', 'Hard-B']):
    model = BoundaryEstimationModel(**EXPERT_PARAMS)
    model.reset_belief(burn_in=100)
    
    rng = np.random.default_rng(42)
    stim = sample_stimuli(1000, dist_name, rng)
    cats = stimulus_to_category(stim)
    ch, pB = model.simulate_session(stim, cats, rng=rng)
    
    valid = ~np.isnan(ch)
    
    # Stimulus histogram
    ax2 = ax.twinx()
    ax2.hist(stim, bins=20, alpha=0.15, color='grey', density=True)
    ax2.set_ylabel('Stimulus density', color='grey', fontsize=8)
    ax2.tick_params(axis='y', colors='grey', labelsize=7)
    
    # Psychometric
    psych = fit_psychometric(stim[valid], ch[valid])
    if psych['success']:
        y_fit = cumulative_gaussian(x_fine, psych['mu'], psych['sigma'],
                                     psych['lapse_low'], psych['lapse_high'])
        ax.plot(x_fine, y_fit, 'k-', linewidth=2)
    
    # Binned data
    n_bins = 10
    be = np.linspace(-1, 1, n_bins + 1)
    mids = (be[:-1] + be[1:]) / 2
    bi = np.clip(np.digitize(stim[valid], be) - 1, 0, n_bins - 1)
    p_binned = [ch[valid][bi == b].mean() if (bi == b).sum() > 0 else np.nan for b in range(n_bins)]
    n_binned = [(bi == b).sum() for b in range(n_bins)]
    ax.scatter(mids, p_binned, s=[n for n in n_binned], c='k', zorder=3)
    
    ax.axhline(0.5, color='grey', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Stimulus')
    ax.set_ylabel('P(choose B)')
    correct_i = (ch == cats)
    valid_i = ~np.isnan(ch)
    ax.set_title(f'{dist_name}\n(acc={correct_i[valid_i].mean():.0%})')
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Same parameters, different distributions', fontsize=12)
plt.tight_layout()
plt.show()

---

**Next:** `2_stat_param_sensitivity.ipynb` — Which summary statistics are informative
about which model parameters, and which stats track learning?